# Exercise 10 – Sparse VARX: Adding Environmental Drivers
### Workshop Tutorial: From Lotka–Volterra to Real Ecological Data

In [ ]:
knitr::opts_chunk$set(
  echo    = TRUE,
  message = FALSE,
  warning = FALSE
)

------------------------------------------------------------------------

# Background: What is a VARX model?

In Exercise 09 we worked with **sparse VAR** models — models where the
only predictors are the system's own past values (*endogenous*
variables). But in ecology, populations do not evolve in isolation.
Temperature, salinity, oxygen, and nutrients all shape species dynamics
from the *outside*.

A **VARX** model extends VAR by including additional *exogenous*
variables:

$$Y_t = \underbrace{\Phi_1 Y_{t-1} + \dots + \Phi_p Y_{t-p}}_{\text{endogenous (VAR) part}} + \underbrace{B_1 X_{t-1} + \dots + B_s X_{t-s}}_{\text{exogenous (X) part}} + \varepsilon_t$$

where:

| Symbol   | Meaning                                                    |
|----------|------------------------------------------------------------|
| $Y_t$    | Endogenous variables at time $t$ (e.g. taxon abundances)   |
| $X_t$    | Exogenous variables at time $t$ (e.g. temperature, oxygen) |
| $\Phi_l$ | Autoregressive coefficient matrices for lag $l$            |
| $B_l$    | Exogenous effect matrices for lag $l$                      |
| $p, s$   | Maximum lag orders for endogenous and exogenous variables  |

The key advantage: environmental forcing can be captured *directly*
rather than absorbed as noise, leading to better forecasts and more
interpretable ecological networks.

------------------------------------------------------------------------

# Exercise 1 — Recovering Lotka–Volterra Parameters with VARX

## Learning Goals

You will connect a **continuous-time ecological model** (generalized
Lotka–Volterra, gLV) to a **discrete-time VARX representation** and
estimate its parameters from simulated data. This is a key conceptual
bridge: real ecological interaction networks are often described as gLV
systems, and sparse VARX gives us a way to *infer* those networks from
time series.

------------------------------------------------------------------------

## The Generalized Lotka–Volterra System

The gLV model describes the dynamics of $d$ interacting species:

$$\frac{dx_i}{dt} = r_i x_i + \sum_{j=1}^{d} \alpha_{ij}\, x_i\, x_j, \quad i = 1, \dots, d$$

In matrix form:
$$\frac{dx}{dt} = \mathrm{diag}(x)\left(r + \alpha x\right)$$

Parameter meanings:

| Parameter | Meaning |
|----|----|
| $r_i > 0$ | Species $i$ grows intrinsically (e.g. prey) |
| $r_i < 0$ | Species $i$ declines intrinsically (e.g. predator without prey) |
| $\alpha_{ij} > 0$ | Species $j$ *promotes* growth of species $i$ |
| $\alpha_{ij} < 0$ | Species $j$ *inhibits* species $i$ |

**From gLV to VARX:** Taking the log-difference of the gLV system yields
an approximate discrete-time representation:
$$\Delta \log x_i(t) \approx r_i \cdot \Delta t + \sum_j \alpha_{ij}\, x_j(t-1) \cdot \Delta t + \varepsilon$$

This is exactly a **VARX(1,1)** model where:

-   The **endogenous** variable is $\Delta \log x$ (the log-difference
    of abundances)
-   The **exogenous** variable is $x$ itself (the raw abundances at the
    previous step)
-   The exogenous coefficient $B$ encodes the interaction matrix
    $\alpha$, scaled by $\Delta t$
-   The intercept $\phi_0$ encodes the growth rate $r$, scaled by
    $\Delta t$

------------------------------------------------------------------------

## Step 1: Define the true system parameters

We simulate a 3-species predator–prey system: one prey species and two
predators. The true parameter values define the reference against which
we will evaluate our estimates.

In [ ]:
# True parameters of the generating system
parms <- c(
  alpha  =  0.5,   # intrinsic growth rate of prey
  beta1  = -0.5,   # effect of predator 1 on prey (negative = suppression)
  beta2  = -0.1,   # effect of predator 2 on prey (weaker suppression)
  delta1 =  0.25,  # benefit of prey to predator 1 (positive = gains from eating)
  delta2 =  0.2,   # benefit of prey to predator 2
  gamma1 = -1.0,   # intrinsic mortality of predator 1
  gamma2 = -0.8    # intrinsic mortality of predator 2
)

------------------------------------------------------------------------

## Step 2: Organise the true parameters into r and α

We separate the parameters into the two components that VARX will try to
recover: the growth rate vector $r$ and the interaction matrix $\alpha$.

In [ ]:
# Growth rate vector (one entry per species)
r_true <- c(
  Prey  = parms["alpha"],
  Pred1 = parms["gamma1"],
  Pred2 = parms["gamma2"]
)

# Interaction matrix (3 × 3): most entries are zero → sparse system
alpha_true <- matrix(0, nrow = 3, ncol = 3,
                     dimnames = list(c("Prey", "Pred1", "Pred2"),
                                     c("Prey", "Pred1", "Pred2")))

alpha_true["Prey",  "Pred1"] <- parms["beta1"]   # predator 1 suppresses prey
alpha_true["Prey",  "Pred2"] <- parms["beta2"]   # predator 2 suppresses prey
alpha_true["Pred1", "Prey"]  <- parms["delta1"]  # prey benefits predator 1
alpha_true["Pred2", "Prey"]  <- parms["delta2"]  # prey benefits predator 2

cat("True growth rates (r):\n")
print(r_true)

cat("\nTrue interaction matrix (alpha):\n")
print(alpha_true)

> **What to notice:** The diagonal is zero (no self-interaction in this
> model) and most off-diagonal entries are also zero. The true system is
> *sparse* — only prey-predator pairs interact. This sparsity is exactly
> what sparse VARX should recover.

------------------------------------------------------------------------

## Step 3: Load the simulated time series

We load a pre-simulated trajectory from this Lotka–Volterra system. Each
column is one species; rows are time points.

In [ ]:
library(bigtime)

Y <- read.csv("lotka_volterra_sim_2.csv", check.names = FALSE)

cat("Loaded time series:\n")
cat("  Time points:", nrow(Y), "\n")
cat("  Species:    ", ncol(Y), "\n")
cat("  Columns:    ", paste(colnames(Y), collapse = ", "), "\n")

------------------------------------------------------------------------

## Step 4: Construct the endogenous and exogenous matrices

This is the key data preparation step that translates the gLV system
into VARX form.

-   **Y_endog** = $\Delta \log(x)$ — the *change* in log-abundance from
    one step to the next
-   **X_exog** = $x$ at the previous time point — the raw abundances
    used as the exogenous predictor

The time step $\Delta t$ is needed later to recover the original gLV
parameters from the estimated VARX coefficients.

In [ ]:
dt <- 0.2        # time step used in the simulation

n       <- nrow(Y)
Y_log   <- log(Y)                       # log-transform abundances

Y_endog <- diff(as.matrix(Y_log))       # first differences of log: T-1 × k
X_exog  <- as.matrix(Y[1:(n - 1), ])   # lagged raw abundances:   T-1 × k

colnames(Y_endog) <- c("logPrey", "logPred1", "logPred2")
colnames(X_exog)  <- c("Prey",    "Pred1",    "Pred2")

cat("Y_endog (log-differences):", nrow(Y_endog), "×", ncol(Y_endog), "\n")
cat("X_exog  (raw abundances): ", nrow(X_exog),  "×", ncol(X_exog),  "\n")

------------------------------------------------------------------------

## Step 5: Fit the sparse VARX model

### Key function: `sparseVARX()`

`sparseVARX()` from the `bigtime` package estimates a regularized VARX
model. It penalizes both the endogenous coefficient matrix $\Phi$ and
the exogenous coefficient matrix $B$ separately, each with its own
penalty parameter.

| Argument | Description |
|----|----|
| `Y` | Endogenous time series matrix (T × k) |
| `X` | Exogenous time series matrix (T × q) — must have the same number of rows as `Y` |
| `p` | Maximum lag order for the endogenous part ($\Phi$ matrices) |
| `s` | Maximum lag order for the exogenous part ($B$ matrices) |
| `VARXpen` | Penalty type: `"L1"` (LASSO) or `"HLag"` (Hierarchical Lag) |
| `VARXlPhiseq` | Grid of penalty values (λ) to try for $\Phi$ |
| `VARXlBseq` | Grid of penalty values (λ) to try for $B$ |
| `selection` | How to select the optimal λ: `"cv"` for cross-validation |

The fitted object contains:

| Field | Description |
|----|----|
| `$Phihat` | Estimated endogenous coefficient matrix (k × k·p) |
| `$Bhat` | Estimated exogenous coefficient matrix (k × q·s) |
| `$phi0hat` | Estimated intercept vector (length k) — encodes growth rates |
| `$MSFEcv` | Cross-validated mean squared forecast errors |
| `$lambdaPhi_opt` | Optimal λ for the endogenous part |
| `$lambdaB_opt` | Optimal λ for the exogenous part |

In [ ]:
fit_varx <- sparseVARX(
  Y           = Y_endog,
  X           = X_exog,
  p           = 1,
  s           = 1,
  VARXpen     = "L1",
  VARXlPhiseq = seq(10, 100, length.out = 10),
  VARXlBseq   = seq(10, 100, length.out = 10),
  selection   = "cv"
)

cat("Optimal λ for Phi (endogenous):", fit_varx$lambdaPhi_opt, "\n")
cat("Optimal λ for B   (exogenous): ", fit_varx$lambdaB_opt,  "\n")

------------------------------------------------------------------------

## Step 6: Inspect diagnostics and lag structure

Before interpreting parameters, we verify that the model captures the
temporal structure of the data reasonably well.

In [ ]:
# Residual diagnostics for the third variable (Pred2)
p_cv <- diagnostics_plot(fit_varx, variable = 3)
plot(p_cv)

In [ ]:
# Lag matrix: which lags are active for each endogenous and exogenous variable?
lagmatrix(fit_varx, returnplot = TRUE)

> **Reading the lag matrix:** The lag matrix now has *two* parts — one
> for the endogenous (lagged $\Delta\log x$) effects and one for the
> exogenous (lagged $x$) effects. Since we set $p = s = 1$, there is
> only one lag to inspect. Non-zero entries in the $B$ part directly
> correspond to species interaction terms in the gLV model.

------------------------------------------------------------------------

## Step 7: Recover the gLV parameters from VARX estimates

We now back-transform the fitted VARX coefficients to the original gLV
parameter scale. Recall:

$$\hat{r}_i = \frac{\hat{\phi}_{0,i}}{\Delta t}, \qquad \hat{\alpha} = \frac{\hat{B}}{\Delta t}$$

In [ ]:
# --- Growth rates ---
# phi0hat is the intercept; dividing by dt recovers r
r_hat <- fit_varx$phi0hat / dt

r_compare <- data.frame(
  Species = colnames(Y_endog),
  True_r  = as.numeric(r_true),
  Est_r   = as.numeric(r_hat)
)

cat("Growth rate comparison:\n")
print(r_compare)

In [ ]:
# --- Interaction matrix ---
# Bhat may contain multiple lag blocks (s lags); sum them before rescaling
k <- ncol(X_exog)
q <- fit_varx$s     # number of exogenous lags

B_total <- matrix(0, nrow = k, ncol = k)
for (lag in 1:q) {
  B_lag   <- fit_varx$Bhat[, ((lag - 1) * k + 1):(lag * k)]
  B_total <- B_total + B_lag
}

alpha_hat <- B_total / dt

alpha_compare <- data.frame(
  Interaction = outer(rownames(alpha_true), colnames(alpha_true),
                      paste, sep = " ← ") |> as.vector(),
  True_alpha  = as.vector(alpha_true),
  Est_alpha   = as.vector(alpha_hat)
)

cat("\nInteraction matrix comparison:\n")
print(alpha_compare)

> **Interpretation:**
>
> -   **Signs** are the most important thing to get right. Does the
>     model correctly identify which interactions are positive
>     (mutualism / predation benefit) and which are negative
>     (suppression)?
> -   **Zero vs. non-zero:** Does the model correctly shrink the zero
>     entries in `alpha_true` to near zero?
> -   **Magnitudes** will not match perfectly due to discretization
>     error and regularization, especially for small true values like
>     $\beta_2 = -0.1$.

------------------------------------------------------------------------

# Exercise 2 — Sparse VAR vs. VARX on Baltic Sea Data

## Learning Goals

You will apply both a sparse VAR and two sparse VARX models (LASSO and
HLag penalties) to a real multivariate marine time series. By adding
environmental variables as exogenous drivers, you will evaluate whether
the model captures more of the system's dynamics.

You will compare:

-   **sparse VAR** — endogenous taxa dynamics only
-   **sparse VARX (LASSO)** — with Temperature and Oxygen as exogenous
    predictors, L1 penalty
-   **sparse VARX (HLag)** — same predictors, hierarchical lag penalty

------------------------------------------------------------------------

## Step 1: Load and inspect the data

We load the biological time series and the environmental metadata
separately.

In [ ]:
time_series <- as.matrix(read.csv("time_series_balticsea.csv"))
metadata    <- as.matrix(read.csv("metadata_time_series_imp.csv"))

# Assign meaningful column names to metadata
colnames(metadata) <- c("Temperature", "oxy_ml_per_L", "Chl",
                        "pressure", "NO3_LOD2", "NO2_LOD2",
                        "Silicate_LOD2", "PO4_LOD2")

cat("Biological time series:  ", nrow(time_series), "time points ×",
    ncol(time_series), "taxa\n")
cat("Environmental metadata:  ", nrow(metadata),    "time points ×",
    ncol(metadata),    "variables\n")
cat("Environmental variables: ", paste(colnames(metadata), collapse = ", "), "\n")

> **Endogenous vs. exogenous — a reminder:**
>
> -   **Endogenous (Y):** The taxa abundances. These are the variables
>     whose dynamics we want to model and forecast.
> -   **Exogenous (X):** Temperature and oxygen. These are external
>     drivers that influence the taxa but are not themselves explained
>     by the model.
>
> Scaling both matrices to zero mean and unit variance is important for
> regularized models: without scaling, variables with larger absolute
> values would need smaller penalties to be retained, making the
> regularization unequal across predictors.

------------------------------------------------------------------------

## Step 2: Fit the LASSO-penalized sparse VARX

We fit a VARX with $p = 3$ lags for the endogenous part and use
Temperature and Oxygen as exogenous variables. The penalty grid
`VARXlBseq` controls the range of λ values tested for the exogenous
coefficient matrix $B$.

In [ ]:
fit_varx <- sparseVARX(
  Y         = scale(time_series),
  X         = scale(metadata[, c("Temperature", "oxy_ml_per_L")]),
  p         = 3,
  VARXpen   = "L1",
  VARXlBseq = seq(0.1, 10, length.out = 10),
  selection = "cv"
)

cat("Optimal λ for Phi (endogenous):", fit_varx$lambdaPhi_opt, "\n")

------------------------------------------------------------------------

## Step 3: Extract the optimal CV-MSFE for the LASSO VARX

The model stores a grid of cross-validated MSFE values, one for every
combination of $\lambda_\Phi$ and $\lambda_B$. We extract the MSFE at
the model selected by the one-standard-error rule.

In [ ]:
# Find grid point matching the SE-optimal penalty combination
idx_lasso   <- which(fit_varx$lambdaB   == fit_varx$lambdaB_SEopt &
                     fit_varx$lambdaPhi == fit_varx$lambdaPhi_SEopt)
best_lasso  <- idx_lasso[which.min(fit_varx$MSFEcv[idx_lasso])]
cv_lasso    <- fit_varx$MSFEcv[best_lasso]

cat("CV-MSFE (LASSO VARX):", round(cv_lasso, 4), "\n")

> **What is the 1-SE rule?** Instead of choosing the λ that gives the
> absolute minimum CV error, the 1-SE rule selects the *simplest* model
> (largest λ, most sparse) whose CV error is within one standard error
> of the minimum. This prevents over-reliance on marginal improvements
> in the CV curve.

------------------------------------------------------------------------

## Step 4: Diagnostics and lag structure — LASSO VARX

In [ ]:
diagnostics_plot(fit_varx)

In [ ]:
l_varx <- lagmatrix(fit_varx, returnplot = TRUE)

> **Reading the VARX lag matrix:** The lag matrix now has two sections:
>
> -   **LPhi** — active lags among the taxa themselves (endogenous part)
> -   **LB** — active lags from the exogenous variables (Temperature,
>     Oxygen) onto each taxon

------------------------------------------------------------------------

## Step 5: Fit the HLag-penalized sparse VARX

We fit the same model structure but switch to the `"HLag"` penalty.
Recall from Exercise 09 that HLag enforces a nested lag structure: if a
variable is active at Lag 3, it must also be active at Lags 1 and 2.

In [ ]:
fit_varx_h <- sparseVARX(
  Y         = scale(time_series),
  X         = scale(metadata[, c("Temperature", "oxy_ml_per_L")]),
  p         = 3,
  VARXpen   = "HLag",
  VARXlBseq = seq(0.1, 10, length.out = 10),
  selection = "cv"
)

cat("Optimal λ for Phi (endogenous):", fit_varx_h$lambdaPhi_opt, "\n")

------------------------------------------------------------------------

## Step 6: Extract CV-MSFE for HLag VARX

In [ ]:
idx_hlag  <- which(fit_varx_h$lambdaB   == fit_varx_h$lambdaB_SEopt &
                   fit_varx_h$lambdaPhi == fit_varx_h$lambdaPhi_SEopt)
best_hlag <- idx_hlag[which.min(fit_varx_h$MSFEcv[idx_hlag])]
cv_hlag   <- fit_varx_h$MSFEcv[best_hlag]

cat("CV-MSFE (HLag VARX):", round(cv_hlag, 4), "\n")

------------------------------------------------------------------------

## Step 7: Diagnostics and lag structure — HLag VARX

In [ ]:
diagnostics_plot(fit_varx_h)

In [ ]:
l_varx_h <- lagmatrix(fit_varx_h, returnplot = TRUE)

> **Compare LASSO vs HLag lag matrices:** Does HLag produce a more
> "staircase-like" pattern where consecutive lags tend to be active
> together? LASSO may show isolated active lags (e.g. Lag 3 active but
> not Lag 1), which can be harder to interpret ecologically.

------------------------------------------------------------------------

## Step 8: Fit a sparse VAR (no exogenous variables)

As a baseline, we fit a standard sparse VAR using only the endogenous
taxa — exactly as in Exercise 09. This lets us quantify how much the
exogenous variables help.

In [ ]:
fit_var <- sparseVAR(
  Y         = scale(time_series),
  p         = 3,
  selection = "cv"
)

------------------------------------------------------------------------

## Step 9: Extract CV-MSFE for the VAR baseline

In [ ]:
# sparseVAR uses a single lambda sequence (not a 2D grid like VARX)
plot_cv(fit_var)

idx_var  <- which(fit_var$lambdas == fit_var$lambda_opt)
best_var <- idx_var[which.min(fit_var$MSFEcv[idx_var])]
cv_var   <- fit_var$MSFEcv[best_var]

cat("CV-MSFE (sparse VAR, no exogenous):", round(cv_var, 4), "\n")

------------------------------------------------------------------------

## Step 10: Diagnostics and lag structure — VAR baseline

In [ ]:
diagnostics_plot(fit_var)

In [ ]:
l_var <- lagmatrix(fit_var, returnplot = TRUE)

------------------------------------------------------------------------

## Step 11: Compare model complexity (non-zero coefficients)

More non-zero coefficients = more complex model. We compare the sparsity
of the endogenous part across all three models, and additionally count
the exogenous coefficients in the VARX models.

In [ ]:
nz_var       <- sum(fit_var$Phihat  != 0)
nz_varx_phi  <- sum(fit_varx$Phihat != 0)
nz_varx_B    <- sum(fit_varx$Bhat   != 0)

cat("Non-zero coefficients:\n")
cat(sprintf("  sparse VAR   — Phi:           %d\n", nz_var))
cat(sprintf("  LASSO VARX   — Phi:           %d\n", nz_varx_phi))
cat(sprintf("  LASSO VARX   — B (exogenous): %d\n", nz_varx_B))

> **Why might VARX have *fewer* non-zero Phi coefficients than VAR?**
> When temperature or oxygen explains part of a taxon's variability, the
> endogenous model no longer needs spurious taxon–taxon links to account
> for that variance. Including relevant exogenous variables can
> therefore *purify* the inferred ecological network by removing
> confounding environmental effects.

------------------------------------------------------------------------

## Step 12: Compare lag matrices — VAR vs. VARX

We subtract the two Phi lag matrices to highlight which taxon–taxon
interactions are present in one model but not the other.

In [ ]:
# Heatmap of differences in active lags
heatmap(
  as.matrix(l_var$LPhi) - as.matrix(l_varx$LPhi),
  main   = "Δ Active lags: VAR minus LASSO VARX",
  xlab   = "Predictor taxon",
  ylab   = "Response taxon",
  scale  = "none",
  col    = colorRampPalette(c("#b2182b", "white", "#2166ac"))(50)
)

> **Reading this plot:** Blue cells = lags active in VAR but not in
> VARX. These are endogenous links that disappear once environmental
> covariates are included — suggesting they were proxies for
> environmental effects rather than true biological interactions.

------------------------------------------------------------------------

## Step 13: Coefficient heatmaps

We visualize the magnitude of estimated coefficients for: (a)
taxon–taxon interactions, (b) the effect of Temperature, and (c) the
effect of Oxygen.

### Helper function: `heatmap_coef()`

This is a custom plotting function that creates a `ggplot2` tile heatmap
from a coefficient matrix.

In [ ]:
library(ggplot2)
library(tidyr)
library(dplyr)
library(tibble)
library(patchwork)

heatmap_coef <- function(mat, title, show_y = TRUE, fixed = TRUE) {
  p <- as.data.frame(mat) |>
    rownames_to_column("response") |>
    pivot_longer(-response, names_to = "predictor", values_to = "coef") |>
    ggplot(aes(x = predictor, y = response, fill = abs(coef))) +
    geom_tile(color = "grey80", linewidth = 0.2) +
    geom_text(
      aes(label = ifelse(abs(coef) > 0.01, round(abs(coef), 1), "")),
      size = 2, color = "grey20"
    ) +
    scale_fill_gradient(
      low  = "white", high = "#08519c", name = NULL,
      guide = guide_colorbar(barwidth = 6, barheight = 0.4, ticks = FALSE)
    ) +
    scale_x_discrete(position = "top") +
    scale_y_discrete(limits = rev) +
    theme_minimal(base_size = 8) +
    theme(
      axis.text.x     = element_text(angle = 90, vjust = 0.5, hjust = 0, size = 6),
      axis.text.y     = element_text(size = 6),
      axis.title      = element_blank(),
      panel.border    = element_rect(color = "black", fill = NA, linewidth = 0.8),
      panel.grid      = element_blank(),
      legend.position = "top",
      plot.title      = element_text(size = 8, hjust = 0.5)
    ) +
    labs(title = title)

  if (fixed)    p <- p + coord_fixed()
  if (!show_y)  p <- p + theme(axis.text.y = element_blank())
  p
}

### LASSO VARX coefficient heatmaps

In [ ]:
# (a) Taxon–taxon interactions (endogenous Phi matrix)
p1 <- heatmap_coef(l_varx$LPhi, "(a) LASSO: Taxa interactions (endogenous)", fixed = TRUE)

# (b) Temperature effects — extract column 1 from LB
mat_temp <- l_varx$LB[, 1, drop = FALSE]
colnames(mat_temp) <- "Temperature"
p2 <- heatmap_coef(mat_temp, "(b) LASSO: Temperature effect (exogenous)", fixed = FALSE)

# (c) Oxygen effects — extract column 2 from LB
mat_oxy <- l_varx$LB[, 2, drop = FALSE]
colnames(mat_oxy) <- "Oxygen"
p3 <- heatmap_coef(mat_oxy, "(c) LASSO: Oxygen effect (exogenous)", fixed = FALSE)

p1
p2
p3

### HLag VARX coefficient heatmaps

In [ ]:
p1_h <- heatmap_coef(l_varx_h$LPhi, "(a) HLag: Taxa interactions (endogenous)", fixed = TRUE)

mat_temp_h <- l_varx_h$LB[, 1, drop = FALSE]
colnames(mat_temp_h) <- "Temperature"
p2_h <- heatmap_coef(mat_temp_h, "(b) HLag: Temperature effect (exogenous)", fixed = FALSE)

mat_oxy_h <- l_varx_h$LB[, 2, drop = FALSE]
colnames(mat_oxy_h) <- "Oxygen"
p3_h <- heatmap_coef(mat_oxy_h, "(c) HLag: Oxygen effect (exogenous)", fixed = FALSE)

p1_h
p2_h
p3_h

------------------------------------------------------------------------

## Step 14: In-sample MSE comparison

In-sample MSE measures how well each model fits the *training data*
(lower = better fit, but can indicate overfitting).

### Key function: `residuals()`

`residuals(fit)` extracts the model residuals (observed minus fitted
values). We compute the mean squared residual as a measure of in-sample
fit.

In [ ]:
mse_var   <- mean(residuals(fit_var)^2)
mse_varx  <- mean(residuals(fit_varx)^2)

cat("In-sample MSE:\n")
cat(sprintf("  sparse VAR  : %.4f\n", mse_var))
cat(sprintf("  LASSO VARX  : %.4f\n", mse_varx))

------------------------------------------------------------------------

## Step 15: Residual ACF comparison

We plot the autocorrelation function (ACF) of residuals for the first
taxon in each model. If the model has captured the temporal structure,
residuals should resemble white noise (no significant autocorrelations
beyond lag 0).

In [ ]:
par(mfrow = c(1, 2))

acf(residuals(fit_var)[, 1],
    main = "ACF of residuals: sparse VAR (taxon 1)",
    lag.max = 20)

acf(residuals(fit_varx)[, 1],
    main = "ACF of residuals: LASSO VARX (taxon 1)",
    lag.max = 20)

par(mfrow = c(1, 1))

------------------------------------------------------------------------

## Step 16: Ljung–Box test for white-noise residuals

The Ljung–Box test formally tests whether residuals show significant
autocorrelation. The null hypothesis is "no autocorrelation" (i.e. the
residuals are white noise).

### Key function: `Box.test()`

| Argument | Description |
|----|----|
| `x` | A univariate time series (single column of residuals) |
| `lag` | Number of lags to include in the test statistic. Rule of thumb: \~$\log(T)$ or $p + 1$ |
| `type` | `"Ljung-Box"` is the standard choice for diagnostic checking |

A **p-value \> 0.05** means we *cannot* reject white noise → residuals
look clean for that variable.

In [ ]:
check_residuals <- function(res_matrix, model_name) {
  p_values <- apply(res_matrix, 2, function(x) {
    Box.test(x, lag = 10, type = "Ljung-Box")$p.value
  })

  data.frame(
    Model          = model_name,
    Variable       = colnames(res_matrix),
    P_Value        = round(p_values, 4),
    Is_White_Noise = p_values > 0.05
  )
}

test_var  <- check_residuals(residuals(fit_var),   "VAR")
test_varx <- check_residuals(residuals(fit_varx),  "VARX")

comparison_table <- rbind(test_var, test_varx)
print(comparison_table)

In [ ]:
cat("Variables with white-noise residuals:\n")
cat(sprintf("  sparse VAR  : %d / %d\n",
            sum(comparison_table$Is_White_Noise[comparison_table$Model == "VAR"]),
            ncol(time_series)))
cat(sprintf("  LASSO VARX  : %d / %d\n",
            sum(comparison_table$Is_White_Noise[comparison_table$Model == "VARX"]),
            ncol(time_series)))

> **Discussion questions:**
>
> 1.  Does VARX have more white-noise residuals than VAR? What does this
>     tell us about the role of environmental drivers?
> 2.  Are there specific taxa where both models fail (non-white
>     residuals)? What might cause this?
> 3.  Which model would you choose for forecasting? For network
>     reconstruction? Are these the same answer?

------------------------------------------------------------------------

# Exercise 3 — Exploring Additional Covariates

## Learning Goals

You will extend the VARX model by including additional environmental
variables (beyond Temperature and Oxygen) and evaluate systematically
how each covariate contributes to model performance, sparsity, and
interpretability.

------------------------------------------------------------------------

## Step 1: Fit a VARX with three covariates (Temperature, Oxygen, Chlorophyll)

Chlorophyll (Chl) is a proxy for primary productivity — the energy base
of the food web. We add it as a third exogenous variable and compare the
result to the two-covariate model from Exercise 2.

In [ ]:
fit_varx_chl <- sparseVARX(
  Y         = scale(time_series),
  X         = scale(metadata[, c("Temperature", "oxy_ml_per_L", "Chl")]),
  p         = 3,
  selection = "cv",
  VARXlBseq = seq(0.1, 10, length.out = 10)
)

------------------------------------------------------------------------

## Step 2: CV diagnostics for the three-covariate model

In [ ]:
plot_cv(fit_varx_chl)

------------------------------------------------------------------------

## Step 3: Extract optimal CV-MSFE

In [ ]:
fit_varx_chl$lambdaPhi_opt

idx_chl  <- which(fit_varx_chl$lambdaB   == fit_varx_chl$lambdaB_SEopt &
                  fit_varx_chl$lambdaPhi == fit_varx_chl$lambdaPhi_SEopt)
best_chl <- idx_chl[which.min(fit_varx_chl$MSFEcv[idx_chl])]
cv_chl   <- fit_varx_chl$MSFEcv[best_chl]

cat("CV-MSFE (VARX with Temp + Oxy + Chl):", round(cv_chl, 4), "\n")
cat("CV-MSFE (VARX with Temp + Oxy only): ", round(cv_lasso, 4), "\n")
cat("\nDifference:", round(cv_chl - cv_lasso, 4),
    "(negative = adding Chl improves prediction)\n")

------------------------------------------------------------------------

## Step 4: Residual diagnostics and lag structure

In [ ]:
diagnostics_plot(fit_varx_chl)

In [ ]:
l_varx_chl <- lagmatrix(fit_varx_chl, returnplot = TRUE)

------------------------------------------------------------------------

## Step 5: Summary comparison across all models

We collect the key metrics from all four models in a single comparison
table.

In [ ]:
# Non-zero coefficients
nz_varx_h_phi <- sum(fit_varx_h$Phihat != 0)
nz_varx_h_B   <- sum(fit_varx_h$Bhat   != 0)
nz_chl_phi    <- sum(fit_varx_chl$Phihat != 0)
nz_chl_B      <- sum(fit_varx_chl$Bhat   != 0)

summary_table <- data.frame(
  Model             = c("sparse VAR", "VARX LASSO (T+O)", "VARX HLag (T+O)", "VARX LASSO (T+O+Chl)"),
  CV_MSFE           = round(c(cv_var, cv_lasso, cv_hlag, cv_chl), 4),
  Nonzero_Phi       = c(nz_var, nz_varx_phi, nz_varx_h_phi, nz_chl_phi),
  Nonzero_B         = c(NA,     nz_varx_B,   nz_varx_h_B,   nz_chl_B),
  WhiteNoise_resid  = c(
    sum(check_residuals(residuals(fit_var),       "VAR")$Is_White_Noise),
    sum(check_residuals(residuals(fit_varx),      "VARX")$Is_White_Noise),
    sum(check_residuals(residuals(fit_varx_h),    "HVARX")$Is_White_Noise),
    sum(check_residuals(residuals(fit_varx_chl),  "VARX_chl")$Is_White_Noise)
  )
)

print(summary_table)

------------------------------------------------------------------------

## Final Discussion

Use the summary table and your observations to work through the
following questions:

**1. Does adding exogenous variables improve prediction?** Compare
CV-MSFE across models. Is the improvement consistent? Does adding
Chlorophyll on top of Temperature and Oxygen still help?

**2. Effect on sparsity** Does VARX have fewer or more non-zero Phi
entries than VAR? Why might exogenous variables *reduce* the number of
endogenous links needed?

**3. Lagged environmental effects** From the lag matrices, are
Temperature and Oxygen effects immediate (Lag 1) or delayed (Lag 2, 3)?
What biological mechanism could cause a lagged response?

**4. Overfitting risk** At what point does adding more covariates stop
improving (or even hurt) CV-MSFE? What does this tell us about covariate
selection in practice?

**5. LASSO vs. HLag for exogenous effects** Compare the heatmaps for
Temperature and Oxygen between LASSO and HLag VARX. Does HLag produce a
cleaner, more ecologically interpretable pattern?

**6. VAR vs. VARX for network reconstruction** Which model would you
trust more as a representation of the *true* biological interaction
network? Is a link in the VAR Phi matrix necessarily a direct biological
interaction, or could it be a shared environmental response?

------------------------------------------------------------------------

*End of Exercise 10*